In [1]:
import pandas as pd
import pickle
import numpy as np
import talib

In [2]:
with open('../../data/nifty_data.pkl', 'rb') as file:
    nifty_data = pickle.load(file)

In [8]:
nifty_data.tail()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-09-15 00:00:00+05:30,25118.900391,25138.449219,25048.750000,25069.199219,185400,0.0,0.0
2025-09-16 00:00:00+05:30,25073.599609,25261.400391,25070.449219,25239.099609,240100,0.0,0.0
2025-09-17 00:00:00+05:30,25276.599609,25346.500000,25275.349609,25330.250000,268900,0.0,0.0
2025-09-18 00:00:00+05:30,25441.050781,25448.949219,25329.750000,25423.599609,272200,0.0,0.0
2025-09-19 00:00:00+05:30,25410.199219,25428.750000,25286.300781,25327.050781,380400,0.0,0.0


In [9]:
df = nifty_data.copy()

In [10]:
df["log_return_1"]  = np.log(df["Close"] / df["Close"].shift(1))
df["log_return_5"]  = np.log(df["Close"] / df["Close"].shift(5))
df["log_return_20"] = np.log(df["Close"] / df["Close"].shift(20))

log_ret = np.log(df["Close"] / df["Close"].shift(1))

df["realized_vol_5"]  = np.sqrt((log_ret ** 2).rolling(5).sum())
df["realized_vol_20"] = np.sqrt((log_ret ** 2).rolling(20).sum())
df["realized_vol_60"] = np.sqrt((log_ret ** 2).rolling(60).sum())

df["true_range"] = talib.TRANGE(
    df["High"].values,
    df["Low"].values,
    df["Close"].values
)

range_ratio = (df["High"] - df["Low"]) / df["Close"]
df["range_ratio_20"] = range_ratio.rolling(20).mean()

def rolling_max_drawdown(close, window):
    rolling_max = close.rolling(window).max()
    drawdown = close / rolling_max - 1.0
    return drawdown.rolling(window).min()

df["rolling_max_dd_20"] = rolling_max_drawdown(df["Close"], 20)
df["rolling_max_dd_60"] = rolling_max_drawdown(df["Close"], 60)

def rolling_autocorr(series, window, lag=1):
    return series.rolling(window).apply(
        lambda x: x.autocorr(lag=lag),
        raw=False
    )

df["autocorr_5"]  = rolling_autocorr(log_ret, 5)
df["autocorr_20"] = rolling_autocorr(log_ret, 20)


In [11]:
FEATURES = [
    "log_return_1", "log_return_5", "log_return_20",
    "realized_vol_5", "realized_vol_20", "realized_vol_60",
    "true_range", "range_ratio_20",
    "rolling_max_dd_20", "rolling_max_dd_60",
    "autocorr_5", "autocorr_20",
    "Close"
]

In [12]:
df_feat = df[FEATURES].dropna()

In [13]:
temp_copy = df_feat.copy()
df_feat = df_feat.drop("Close", axis=1)

In [14]:
df_feat

,log_return_1,log_return_5,log_return_20,realized_vol_5,realized_vol_20,realized_vol_60,true_range,range_ratio_20,rolling_max_dd_20,rolling_max_dd_60,autocorr_5,autocorr_20
Date,,,,,,,,,,,,
2008-03-05 00:00:00+05:30,0.011680,-0.068134,-0.078359,0.058637,0.112088,0.212806,89.500000,0.031581,-0.200395,-0.230540,-0.044841,0.266042
2008-03-07 00:00:00+05:30,-0.030911,-0.102210,-0.073057,0.066210,0.110489,0.213841,249.149902,0.031911,-0.200395,-0.241140,-0.275571,0.231294
2008-03-10 00:00:00+05:30,0.006018,-0.084468,-0.064524,0.065441,0.110625,0.213686,194.450195,0.032574,-0.200395,-0.241140,-0.309373,0.232045
2008-03-11 00:00:00+05:30,0.013552,-0.017742,0.001831,0.040481,0.098150,0.213473,155.649902,0.030850,-0.184897,-0.241140,-0.534968,0.250618
2008-03-12 00:00:00+05:30,0.001253,0.001592,0.006951,0.036241,0.098082,0.213470,164.450195,0.031203,-0.166365,-0.241140,-0.364252,0.262471
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15 00:00:00+05:30,-0.001785,0.011880,0.017622,0.007482,0.023662,0.042633,89.699219,0.006620,-0.026188,-0.049712,-0.646272,0.116114
2025-09-16 00:00:00+05:30,0.006754,0.014788,0.014453,0.009318,0.022518,0.041226,192.201172,0.006659,-0.026188,-0.049712,-0.934544,0.016878
2025-09-17 00:00:00+05:30,0.003605,0.014200,0.013898,0.009068,0.022422,0.041002,107.400391,0.006521,-0.026188,-0.049712,-0.599646,0.038723


In [15]:
mean = df_feat.mean()
std = df_feat.std()

df_feat = (df_feat - mean) / (std + 1e-8)

In [17]:
df_feat

,log_return_1,log_return_5,log_return_20,realized_vol_5,realized_vol_20,realized_vol_60,true_range,range_ratio_20,rolling_max_dd_20,rolling_max_dd_60,autocorr_5,autocorr_20
Date,,,,,,,,,,,,
2008-03-05 00:00:00+05:30,0.888911,-2.450095,-1.446120,2.117724,2.097966,2.537163,-0.473770,2.086251,-2.899183,-1.493367,0.336706,1.299032
2008-03-07 00:00:00+05:30,-2.462517,-3.642956,-1.356650,2.567032,2.045383,2.558045,1.005082,2.125288,-2.899183,-1.620354,-0.179421,1.126135
2008-03-10 00:00:00+05:30,0.443309,-3.021905,-1.212652,2.521409,2.049827,2.554917,0.498394,2.203459,-2.899183,-1.620354,-0.255032,1.129873
2008-03-11 00:00:00+05:30,1.036210,-0.686092,-0.092985,1.040564,1.639607,2.550630,0.138983,1.999977,-2.585613,-1.620354,-0.759672,1.222285
2008-03-12 00:00:00+05:30,0.068391,-0.009300,-0.006577,0.788974,1.637363,2.550564,0.220501,2.041653,-2.210678,-1.620354,-0.377794,1.281265
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15 00:00:00+05:30,-0.170688,0.350824,0.173477,-0.917223,-0.809880,-0.895055,-0.471925,-0.860413,0.625417,0.672985,-1.008651,0.553040
2025-09-16 00:00:00+05:30,0.501291,0.452650,0.120000,-0.808335,-0.847518,-0.923428,0.477561,-0.855890,0.625417,0.672985,-1.653494,0.059274
2025-09-17 00:00:00+05:30,0.253472,0.432056,0.110636,-0.823140,-0.850671,-0.927961,-0.307957,-0.872084,0.625417,0.672985,-0.904351,0.167969


In [18]:
df_feat.describe().T

,count,mean,std,min,25%,50%,75%,max
log_return_1,4300.0,-8.262125e-19,0.999999,-10.970663,-0.441745,0.020464,0.482153,12.822814
log_return_5,4300.0,1.817667e-17,1.000000,-7.514447,-0.484273,0.055676,0.545496,6.787047
log_return_20,4300.0,-6.609700e-18,1.000000,-8.352225,-0.460217,0.056990,0.551311,4.702505
realized_vol_5,4300.0,-5.287760e-17,0.999999,-1.234545,-0.578753,-0.253254,0.217032,8.792385
realized_vol_20,4300.0,-1.586328e-16,1.000000,-1.127709,-0.577409,-0.265272,0.169799,6.740549
realized_vol_60,4300.0,2.908268e-16,1.000000,-0.994518,-0.578363,-0.282388,0.142425,4.466997
true_range,4300.0,-5.287760e-17,1.000000,-1.100418,-0.608200,-0.267899,0.292860,17.060819
range_ratio_20,4300.0,-1.057552e-16,0.999999,-0.987094,-0.551951,-0.274838,0.153044,7.036156
rolling_max_dd_20,4300.0,1.057552e-16,1.000000,-6.358466,-0.303463,0.279201,0.600112,1.098690
rolling_max_dd_60,4300.0,-1.850716e-16,1.000000,-4.166668,-0.222754,0.173124,0.634369,1.093449


In [19]:
df_feat["Close"] = temp_copy["Close"]

In [20]:
with open('../../data/nifty_ts2vec.pkl', 'wb') as file:
    pickle.dump(df_feat, file)